# Évaluation des Modèles — Dakar Power Prediction

Évaluation complète du pipeline LightGBM + LSTM sur `synthetic_data_v2.csv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.model_selection import train_test_split

# Assure que le repo root est dans le path
sys.path.insert(0, str(Path('..').resolve()))
from src.config import MODEL_CONFIG

FEATURES = MODEL_CONFIG['features']
TARGET   = 'coupure'
RANDOM_STATE = 42

print('Features:', FEATURES)

In [ ]:
df = pd.read_csv('../data/synthetic/synthetic_data_v2.csv')
print(f'Shape : {df.shape}')
df.head()

In [ ]:
# Feature engineering — encodage identique à utils_simple.py
if 'date_heure' in df.columns:
    df['date_heure'] = pd.to_datetime(df['date_heure'])
    df['heure']       = df['date_heure'].dt.hour
    df['jour_semaine']= df['date_heure'].dt.dayofweek
    df['mois']        = df['date_heure'].dt.month
    df['saison']      = df['mois'].isin([6, 7, 8]).astype(int)
    df['is_peak_hour']= df['heure'].between(18, 22).astype(int)

X = df[FEATURES]
y = df[TARGET]
print(f'X shape: {X.shape}, target balance: {y.mean():.3f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
with open('../models/lgbm_model.pkl', 'rb') as f:
    lgb_model = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_test_scaled = scaler.transform(X_test)
print('Modèles chargés.')

In [ ]:
y_prob = lgb_model.predict(X_test_scaled)
y_pred = (y_prob >= 0.5).astype(int)

print('=== Classification Report — LightGBM ===')
print(classification_report(y_test, y_pred, target_names=['Pas de coupure', 'Coupure']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Prédit 0', 'Prédit 1'],
    yticklabels=['Réel 0', 'Réel 1']
)
ax.set_title('Matrice de Confusion — LightGBM')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'LightGBM (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('Taux de Faux Positifs')
ax.set_ylabel('Taux de Vrais Positifs')
ax.set_title('Courbe ROC')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print(f'AUC : {roc_auc:.4f}')

In [ ]:
importance = lgb_model.feature_importance(importance_type='gain')
feat_imp = pd.Series(importance, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature Importance — LightGBM (gain)')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.show()

## Conclusion (métriques mesurées, pas estimées)

> ⚠️ Données synthétiques : démonstration technique, pas des mesures du réseau SENELEC.

Chiffres obtenus lors du diagnostic du 17/11/2026 en rechargeant les modèles déployés (`models/`) sur le jeu de test (14 000 lignes, split stratifié 80/20, `random_state=42`, encodage des features **identique au CSV**). Taux de coupure dans le test : **14,05 %**.

| Modèle | AUC-ROC | PR-AUC | Seuil | Précision | Rappel | F1 | Matrice [[VN, FP], [FN, VP]] |
|---|---|---|---|---|---|---|---|
| LightGBM | 0,873 | 0,503 | 0,50 | 0,604 | 0,303 | 0,404 | [[11643, 390], [1371, 596]] |
| LightGBM | | | 0,25 | 0,454 | 0,597 | 0,516 | [[10618, 1415], [792, 1175]] |
| LSTM | 0,874 | 0,513 | 0,50 | 0,627 | 0,239 | 0,346 | [[11753, 280], [1497, 470]] |
| LSTM | | | 0,25 | 0,456 | 0,601 | 0,519 | [[10623, 1410], [785, 1182]] |
| Ensemble (moyenne) | 0,874 | 0,510 | 0,25 | 0,456 | 0,601 | 0,519 | [[10624, 1409], [785, 1182]] |
| Référence « jamais de coupure » | 0,500 | — | — | — | 0 | 0 | [[12033, 0], [1967, 0]] |
| Référence régression logistique | 0,842 | | | | | | |

**Lecture honnête :**
- L'exactitude (accuracy) de LightGBM au seuil 0,5 est de 87,4 %, contre **85,95 %** pour un modèle qui répond toujours « pas de coupure » : le gain réel est faible.
- Au seuil 0,25, environ une alerte sur deux est une fausse alerte.
- Le LSTM n'apporte rien de mesurable par rapport à LightGBM.
- Variables dominantes (gain LightGBM) : `is_peak_hour` (51 %), `humidite_percent` (24 %), `conso_megawatt` (12 %), `temp_celsius` (8 %). Le modèle apprend surtout l'heure de pointe ; à l'intérieur des heures de pointe, l'AUC tombe à 0,68.
- **Biais optimiste :** le jeu de test a servi à l'arrêt anticipé de LightGBM et du LSTM, et au choix du seuil 0,25. Ces chiffres surestiment donc légèrement la performance attendue sur des données nouvelles.
- Toutes ces mesures portent sur des données synthétiques : elles ne disent rien de la performance sur le réseau réel.
